In [1]:
import gdown
import pandas as pd
url1 = 'https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ'
gdown.download(url1, 'data1.csv', quiet=False)
url2 = 'https://drive.google.com/uc?id=1eX9an3tSypvrFn_fiPReYHTca5GpJaJG'
gdown.download(url2, 'data2.csv', quiet=False)

df1 = pd.read_csv('data1.csv')
df2 = pd.read_csv('data2.csv')

Downloading...
From (original): https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ
From (redirected): https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ&confirm=t&uuid=7b22732b-6cde-4aa1-87bc-b5bb9d3e3da3
To: /content/data1.csv
100%|██████████| 794M/794M [00:11<00:00, 70.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eX9an3tSypvrFn_fiPReYHTca5GpJaJG
To: /content/data2.csv
100%|██████████| 6.16k/6.16k [00:00<00:00, 13.5MB/s]
/tmp/ipykernel_3802/2612111364.py:8: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('data1.csv')


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
df = pd.concat([df1, df2], ignore_index=True)
def convert_fa_to_en(text):
    if pd.isna(text):
        return text
    text = str(text)
    fa_digits = '۰۱۲۳۴۵۶۷۸۹'
    en_digits = '0123456789'
    translation_table = str.maketrans(fa_digits, en_digits)
    return text.translate(translation_table)
df['construction_year_en'] = df['construction_year'].apply(convert_fa_to_en)
df['construction_year_clean'] = pd.to_numeric(df['construction_year_en'], errors='coerce')

df['building_size_clean'] = pd.to_numeric(df['building_size'], errors='coerce')
df_clean = df.dropna(subset=['building_size_clean', 'construction_year_clean']).copy()
df_clean = df_clean[(df_clean['building_size_clean'] >= 10) & (df_clean['building_size_clean'] <= 1000)]
new_houses = df_clean[df_clean['construction_year_clean'] >= 1396]['building_size_clean']
old_houses = df_clean[df_clean['construction_year_clean'] < 1396]['building_size_clean']

mean_new = new_houses.mean()
mean_old = old_houses.mean()
std_new = new_houses.std()
std_old = old_houses.std()

print("=== Descriptive Statistics ===")
print(f"New Houses Count: {len(new_houses):,}")
print(f"New Houses Mean Area: {mean_new:.2f} sqm (Std: {std_new:.2f})")
print(f"\nOld Houses Count: {len(old_houses):,}")
print(f"Old Houses Mean Area: {mean_old:.2f} sqm (Std: {std_old:.2f})")
t_stat, p_value = stats.ttest_ind(new_houses, old_houses, equal_var=False, alternative='less')

print("\n=== Hypothesis Testing Results (Welch's t-test) ===")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value}")
alpha = 0.05
if p_value < alpha:
    print("\nConclusion: Reject the null hypothesis (H0).")
    print("There is statistically significant evidence that new houses have a smaller mean area than old houses.")
else:
    print("\nConclusion: Fail to reject the null hypothesis (H0).")
    print("There is not enough evidence to support that new houses are significantly smaller than old houses.")

=== Descriptive Statistics ===
New Houses Count: 393,823
New Houses Mean Area: 127.12 sqm (Std: 99.65)

Old Houses Count: 385,294
Old Houses Mean Area: 104.88 sqm (Std: 86.34)

=== Hypothesis Testing Results (Welch's t-test) ===
t-statistic: 105.3065
p-value: 1.0

Conclusion: Fail to reject the null hypothesis (H0).
There is not enough evidence to support that new houses are significantly smaller than old houses.
